# Chapter 9 — Feature Space: What Does a Linear Model Actually See?

**Book alignment:** PyTorch From First Principles, Chapter 9

**Question this notebook isolates:** Does a learned linear direction recover a planted signal direction (high cosine) even though raw scores, norms, and coefficient magnitudes are not invariant to rescaling?

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(0)
np.random.seed(0)
torch.set_num_threads(1)
print('torch', torch.__version__)

## 1 — nn.Linear transforms the last axis

A `[B, T, D]` tensor holds `B*T` vectors. A per-vector head returns one score per (example, position); one score per sequence needs an explicit aggregation first.

In [ ]:
B, T, D = 2, 4, 8
x = torch.randn(B, T, D)
lin = nn.Linear(D, 1)

scores = lin(x)
print('input', tuple(x.shape), '-> scores', tuple(scores.shape))

one = x[1, 2]
manual = one @ lin.weight[0] + lin.bias[0]
print('manual', float(manual), 'observed', float(scores[1, 2, 0]))

pooled = x.mean(dim=1)
s_pooled = lin(pooled)
s_first = lin(x[:, 0])
print('pooled', tuple(s_pooled.shape), 'first-token', tuple(s_first.shape))
print('mean-pool == first-token scores:', bool(torch.allclose(s_pooled, s_first)))

In [ ]:
assert tuple(scores.shape) == (2, 4, 1)
assert torch.allclose(manual, scores[1, 2, 0])
assert tuple(s_pooled.shape) == (2, 1) and tuple(s_first.shape) == (2, 1)
assert not torch.allclose(s_pooled, s_first)
print('last-axis rule + aggregation choice verified')

## 2 — Dot product, cosine, and distance answer different questions

Scaling a vector changes the dot product but not the cosine. Normalizing along the wrong axis keeps the shape and changes the geometry.

In [ ]:
a = torch.tensor([3.0, 4.0])
b = torch.tensor([1.0, 0.0])
dot = float(a @ b)
cos = float(F.cosine_similarity(a, b, dim=0))
dist = float(torch.linalg.vector_norm(a - b))
print(f'dot={dot} cos={cos:.4f} dist={dist:.4f}')

scales, dots, coss = [], [], []
for s in [0.5, 1.0, 2.0, 10.0]:
    scales.append(s)
    dots.append(float((s * a) @ b))
    coss.append(float(F.cosine_similarity(s * a, b, dim=0)))
print('dots under scaling:', [round(v, 3) for v in dots])
print('cos under scaling :', [round(v, 4) for v in coss])

xb = torch.arange(1.0, 25.0).reshape(2, 4, 3)
n_last = F.normalize(xb, dim=-1)
n_one = F.normalize(xb, dim=1)
row_last = torch.linalg.vector_norm(n_last, dim=-1)
row_one = torch.linalg.vector_norm(n_one, dim=-1)
print('row norms after dim=-1:', [round(float(v), 4) for v in row_last.flatten()])
print('row norms after dim=1 :', [round(float(v), 4) for v in row_one.flatten()])

In [ ]:
assert abs(dot - 3.0) < 1e-5
assert abs(cos - 0.6) < 1e-4
assert torch.allclose(torch.tensor(dots), torch.tensor([1.5, 3.0, 6.0, 30.0]))
assert max(abs(c - 0.6) for c in coss) < 1e-5
assert bool((row_last - 1.0).abs().max() < 1e-5)
assert bool((row_one - 1.0).abs().max() > 1e-3)
print('dot/cosine/distance + normalize-axis verified')

## 3 — Scores are not distances, but a planted direction is recoverable

Rescaling `(w, b)` together leaves predictions and geometric distances fixed while scores move. A linear SVM trained on data with a known direction should recover its direction (cosine near 1) on held-out data.

In [ ]:
w = torch.tensor([0.8, 1.3])
b = torch.tensor(-0.2)
pts = torch.tensor([[2.0, -1.0], [3.0, 2.0], [-2.0, -1.0], [0.0, 0.0]])
s = pts @ w + b
d = s / torch.linalg.vector_norm(w)
s100 = pts @ (100 * w) + 100 * b
d100 = s100 / torch.linalg.vector_norm(100 * w)
pred = (s >= 0).long()
pred100 = (s100 >= 0).long()
print('scores', [round(float(v), 3) for v in s])
print('scores x100', [round(float(v), 1) for v in s100])
print('distances stable:', bool(torch.allclose(d, d100, atol=1e-4)))

class LinearSVM(nn.Module):
    def __init__(self, features):
        super().__init__()
        self.w = nn.Parameter(torch.zeros(features))
        self.b = nn.Parameter(torch.zeros(()))
    def forward(self, x):
        return x @ self.w + self.b

g = torch.Generator().manual_seed(0)
D = 20
true_w = torch.zeros(D)
true_w[3] = 2.0
true_w[7] = -1.5
true_w[15] = 0.8
X = torch.randn(800, D, generator=g)
noise = 0.15 * torch.randn(800, generator=g)
y = torch.where(X @ true_w + noise >= 0, 1.0, -1.0)
Xtr, ytr, Xte, yte = X[:600], y[:600], X[600:], y[600:]

torch.manual_seed(1)
model = LinearSVM(D)
opt = torch.optim.Adam(model.parameters(), lr=0.05)
C = 1.0
for step in range(300):
    opt.zero_grad()
    sc = model(Xtr)
    loss = 0.5 * model.w.square().sum() + C * torch.relu(1 - ytr * sc).mean()
    loss.backward()
    opt.step()

with torch.no_grad():
    te_acc = ((model(Xte) >= 0).float() * 2 - 1 == yte).float().mean().item()
    cos_rec = float(F.cosine_similarity(model.w, true_w, dim=0))
    rnd = torch.randn(D, generator=torch.Generator().manual_seed(9))
    cos_rnd = float(F.cosine_similarity(rnd, true_w, dim=0))
    top3 = list(torch.topk(model.w.abs(), 3).indices.tolist())
print(f'test acc={te_acc:.3f} cosine={cos_rec:.3f} random={cos_rnd:.3f} top3={top3}')

In [ ]:
assert torch.equal(pred, pred100)
assert torch.allclose(d, d100, atol=1e-4)
assert abs(float(s100[0]) / float(s[0]) - 100.0) < 1e-2
assert te_acc > 0.85
assert cos_rec > 0.80
assert cos_rec > abs(cos_rnd) + 0.5
assert set([3, 7, 15]) <= set(top3 + [3, 7, 15][:0]) or len(set(top3) & {3, 7, 15}) >= 2
print('score/distance invariance + direction recovery verified')

## What we earned

Shapes name containers, not objects: `nn.Linear` scores each `D`-vector, dot products mix magnitude with angle, scores move under parameter rescaling while distances do not, and a planted direction is recoverable by cosine even when magnitudes are not.

Chapter 10 keeps the dot product but aims it at other positions: every query vector scored against every key vector, with the same invariance discipline.